# Notebook 04: Layer 3 — MLM Perplexity Evaluation

This is the notebook referenced in the paper setup guide and Section 6 results.
It evaluates SciBERT-based novel tortured phrase detection and produces:

- SciBERT log-perplexity distributions (tortured vs. clean) → Figure 3
- Threshold calibration (τ = 95th percentile of clean corpus)
- Novel candidate detection on the 2024-25 corpus
- **Appendix D**: `data/appendix/appendix_d_novel_candidates.csv`
- Table 5 and Table 6 values for Section 6.2.3

**Prerequisites:**
- `pip install transformers torch` (SciBERT, ~440 MB download on first run)
- `notebooks/01_data_acquisition.ipynb` for the full PubMed corpora
- Without notebook 01, the cells below fall back to a synthetic demo corpus

**Runtime:** ~1-2 hours on CPU for the full 500-paper novel corpus.
For a quick demo, set `MAX_NOVEL_PAPERS = 20` in the config cell.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Configuration ──────────────────────────────────────────────────────────
PERPLEXITY_THRESHOLD_PERCENTILE = 95.0   # calibrate at this clean-corpus percentile
WINDOW_TOKENS                   = 6      # SciBERT sliding window size
STRIDE_TOKENS                   = 3      # sliding window stride
MAX_NOVEL_PAPERS                = 500    # set to 20 for a quick smoke test
TOP_N_CANDIDATES                = 50     # candidates for Appendix D / expert review
# ───────────────────────────────────────────────────────────────────────────

from tpc.layers.mlm_perplexity import PerplexityDetector
from tpc.evaluation.metrics import compute_novel_detection_rate

DATA_DIR     = ROOT / 'data' / 'ground_truth'
APPENDIX_DIR = ROOT / 'data' / 'appendix'
FIGURES_DIR  = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Configuration:')
print(f'  Threshold percentile : {PERPLEXITY_THRESHOLD_PERCENTILE}')
print(f'  Window tokens        : {WINDOW_TOKENS}')
print(f'  Max novel papers     : {MAX_NOVEL_PAPERS}')
print(f'  Top N candidates     : {TOP_N_CANDIDATES}')

## 1. Load corpora

In [ ]:
tortured_path = DATA_DIR / 'confirmed_tortured.tsv'
clean_path    = DATA_DIR / 'confirmed_clean.tsv'

if tortured_path.exists() and clean_path.exists():
    df_pos    = pd.read_csv(tortured_path, sep='\t').head(500)
    df_neg    = pd.read_csv(clean_path,    sep='\t').head(500)
    print(f'PubMed corpora loaded: {len(df_pos)} retracted, {len(df_neg)} clean')
    USE_REAL_CORPUS = True
else:
    print('PubMed corpora not found. Using synthetic demo corpus.')
    print('Run notebooks/01_data_acquisition.ipynb for real results.')
    USE_REAL_CORPUS = False
    tortured_sentences = [
        'The amino corrosive levels were significantly elevated in treated samples.',
        'Profound learning techniques outperformed conventional methods on all benchmarks.',
        'Flag to commotion proportion was computed using standard procedures.',
        'Bosom peril incidence was correlated with age and family history.',
        'Irregular examination of blood samples confirmed the hypothesis.',
        'Fake neural organization architectures were compared systematically.',
        'Nucleic harsh degradation was observed under thermal stress conditions.',
    ] * 20
    clean_sentences = [
        'The amino acid levels were significantly elevated in treated samples.',
        'Deep learning techniques outperformed conventional methods on all benchmarks.',
        'Signal to noise ratio was computed using standard procedures.',
        'Breast cancer incidence was correlated with age and family history.',
        'Random sampling of blood samples confirmed the hypothesis.',
        'Artificial neural network architectures were compared systematically.',
        'Nucleic acid degradation was observed under thermal stress conditions.',
    ] * 20
    df_pos = pd.DataFrame({'abstract': tortured_sentences, 'label': 'retracted'})
    df_neg = pd.DataFrame({'abstract': clean_sentences,    'label': 'clean'})

## 2. Initialize SciBERT perplexity detector

> First run will download `allenai/scibert_scivocab_uncased` (~440 MB).

In [ ]:
print('Loading SciBERT... (downloads ~440 MB on first run)')
detector = PerplexityDetector(
    window_tokens=WINDOW_TOKENS,
    stride_tokens=STRIDE_TOKENS,
    perplexity_threshold=4.5,  # will be re-calibrated in next cell
    max_text_tokens=256,       # truncate for speed; increase for production
)
print('SciBERT loaded.')

## 3. Threshold calibration on clean corpus

Sets τ at the 95th percentile of clean-corpus span perplexities.
This operationalizes the literary-warrant threshold argument from §5.2.3.

In [ ]:
print('Calibrating threshold on clean corpus...')
print('(This processes each clean abstract through SciBERT — allow 10-30 min on CPU)')

clean_texts = df_neg['abstract'].dropna().tolist()[:100]  # use 100 for calibration

tau = detector.calibrate_threshold(
    clean_texts=clean_texts,
    percentile=PERPLEXITY_THRESHOLD_PERCENTILE
)

# Apply the calibrated threshold
detector.perplexity_threshold = tau

print(f'\nCalibrated threshold τ = {tau:.4f}')
print(f'Interpretation: spans with PLL > {tau:.4f} lie outside the')
print(f'95th percentile of legitimate scientific prose.')

## 4. Perplexity distributions: tortured vs. clean (Figure 3)

In [ ]:
print('Computing perplexity distributions...')
print('(Processing abstracts through SciBERT — allow 20-60 min on CPU)')

def get_all_span_perplexities(texts, max_texts=50):
    """Collect per-span PLL values from a list of texts."""
    import torch
    from tpc.layers.mlm_perplexity import _load_model
    tokenizer, model = _load_model(detector.model_name)
    
    all_plls = []
    for text in texts[:max_texts]:
        tokens = tokenizer.tokenize(text.lower())[:detector.max_text_tokens]
        for i in range(0, len(tokens) - detector.window_tokens, detector.stride_tokens):
            span = tokens[i:i + detector.window_tokens]
            pll  = detector._span_log_perplexity(span, tokenizer, model, torch)
            all_plls.append(pll)
    return all_plls

# Collect distributions (use first 50 abstracts from each corpus for speed)
tortured_plls = get_all_span_perplexities(df_pos['abstract'].dropna().tolist(), max_texts=50)
clean_plls    = get_all_span_perplexities(df_neg['abstract'].dropna().tolist(), max_texts=50)

print(f'Tortured spans: {len(tortured_plls)}, mean PLL = {np.mean(tortured_plls):.3f}')
print(f'Clean spans:    {len(clean_plls)},    mean PLL = {np.mean(clean_plls):.3f}')

In [ ]:
# Figure 3: Perplexity distributions
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(clean_plls,    bins=50, alpha=0.6, color='#2166ac',
        label='Clean spans (legitimate scientific prose)', density=True)
ax.hist(tortured_plls, bins=50, alpha=0.6, color='#d73027',
        label='Tortured spans (known violations)', density=True)
ax.axvline(x=detector.perplexity_threshold, color='black', linewidth=2,
           linestyle='--', label=f'Threshold τ = {detector.perplexity_threshold:.2f} (95th pctile clean)')

ax.set_xlabel('SciBERT span log-perplexity (PLL)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(
    'Figure 3: SciBERT perplexity distributions — tortured vs. clean spans\n'
    '(RQ2: Layer 3 statistical/procedural warrant threshold)', fontsize=11
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_perplexity_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Summary stats for paper text
print(f'\nDistribution summary:')
print(f'  Clean  — mean: {np.mean(clean_plls):.3f}, std: {np.std(clean_plls):.3f}, '
      f'95th pctile: {np.percentile(clean_plls, 95):.3f}')
print(f'  Tortured — mean: {np.mean(tortured_plls):.3f}, std: {np.std(tortured_plls):.3f}')
overlap = sum(p < detector.perplexity_threshold for p in tortured_plls) / max(len(tortured_plls), 1)
print(f'  Tortured spans below threshold (missed by L3): {overlap:.1%}')

## 5. Layer 3 evaluation on positive+negative corpus

In [ ]:
from tpc.evaluation.metrics import evaluate_layer_on_corpus

papers_eval = (df_pos.to_dict('records')[:200] + 
               df_neg.to_dict('records')[:200])

print('Evaluating Layer 3 on pos+neg corpus...')
metrics_l3 = evaluate_layer_on_corpus(
    detector=detector,
    papers=papers_eval,
    layer_name='L3_mlm_perplexity',
    corpus_name='positive+negative',
)

print('\n=== Layer 3 Results ===')
for k, v in metrics_l3.to_dict().items():
    if k not in ('layer', 'corpus'):
        print(f'  {k:25s}: {v}')

## 6. Novel candidate detection on 2024-25 corpus (Appendix D)

In [ ]:
novel_path = DATA_DIR / 'novel_2024_2025.tsv'

if novel_path.exists():
    df_novel = pd.read_csv(novel_path, sep='\t').head(MAX_NOVEL_PAPERS)
    print(f'Novel corpus loaded: {len(df_novel)} papers (2024-2025)')
else:
    print('Novel corpus not found (novel_2024_2025.tsv).')
    print('This file is produced by notebook 01 from papers published Jan 2024 - Dec 2025.')
    print('\nFalling back to synthetic novel examples for demo.')
    # Synthetic novel examples: plausible but not in our registry
    novel_texts = [
        'The coagulation cascade was disrupted by elevated fibrin destruction products.',
        'Mitochondrial membrane voltage collapse was observed under oxidative burden.',
        'Gene silencing via little RNA molecules showed promising therapeutic results.',
        'Receptor binding affinity was measured using surface sound resonance.',
        'The gradient descent optimizer achieved convergence after excessive repetitions.',
        'Cellular respiration defects were linked to electron transport series dysfunction.',
        'The attention mechanism in the transformer demonstrated superior memory persistence.',
    ] * (MAX_NOVEL_PAPERS // 7 + 1)
    df_novel = pd.DataFrame({
        'abstract': novel_texts[:MAX_NOVEL_PAPERS],
        'label': 'unlabeled',
        'pmid': [f'SYNTHETIC_{i:04d}' for i in range(MAX_NOVEL_PAPERS)]
    })
    print(f'Synthetic novel corpus: {len(df_novel)} papers')

novel_papers = df_novel.to_dict('records')

In [ ]:
print('Running novel candidate detection...')
print(f'(Processing {len(novel_papers)} papers — allow 30-120 min on CPU for full corpus)')

novel_results = compute_novel_detection_rate(
    novel_papers=novel_papers,
    perplexity_detector=detector,
)

print('\n=== Table 5: Layer 3 Novel Detection Results ===')
table5_rows = [
    ('Total papers screened',               novel_results['total_papers']),
    ('Papers with at least one flagged span', novel_results['papers_with_flags']),
    ('Flag rate',                            f"{novel_results['flag_rate']:.1%}"),
    ('Total candidate spans',               novel_results['total_candidates']),
    ('Unique candidate spans',              novel_results['unique_spans']),
    ('Expert review batch (top N)',         TOP_N_CANDIDATES),
    ('Expert-confirmed (PENDING REVIEW)',   'see appendix_d_novel_candidates.csv'),
    ('Inter-rater kappa (PENDING REVIEW)',  'see instructions below'),
]
for k, v in table5_rows:
    print(f'  {k:45s}: {v}')

## 7. Export Appendix D

In [ ]:
top_candidates = novel_results['top_candidates'][:TOP_N_CANDIDATES]

if top_candidates:
    df_appendix_d = pd.DataFrame(top_candidates)
    # Add columns for expert review (to be filled manually)
    df_appendix_d['rank']                = range(1, len(df_appendix_d) + 1)
    df_appendix_d['proposed_canonical']  = ''   # fill during expert review
    df_appendix_d['domain']              = ''   # fill during expert review
    df_appendix_d['rater_1_assessment']  = ''   # confirmed / false_positive / indeterminate
    df_appendix_d['rater_2_assessment']  = ''   # confirmed / false_positive / indeterminate
    df_appendix_d['recommended_status']  = ''   # candidate / reject
    
    # Reorder columns for readability
    cols = ['rank', 'span', 'log_perplexity', 'confidence', 'paper_pmid',
            'proposed_canonical', 'domain',
            'rater_1_assessment', 'rater_2_assessment', 'recommended_status']
    cols = [c for c in cols if c in df_appendix_d.columns]
    df_appendix_d = df_appendix_d[cols]
    
    output_path = APPENDIX_DIR / 'appendix_d_novel_candidates.csv'
    df_appendix_d.to_csv(output_path, index=False)
    
    print(f'Appendix D saved: {output_path}')
    print(f'{len(df_appendix_d)} candidates for expert review')
    print('\nTop 10 candidates:')
    print(df_appendix_d[['rank','span','log_perplexity','confidence']].head(10).to_string(index=False))
else:
    print('No candidates detected. Try reducing MAX_NOVEL_PAPERS or lower the threshold.')

## 8. Inter-rater agreement (Cohen's κ)

**Instructions for expert review:**
1. Open `data/appendix/appendix_d_novel_candidates.csv`
2. Two reviewers independently fill `rater_1_assessment` and `rater_2_assessment`
   for each row using values: `confirmed`, `false_positive`, `indeterminate`
3. Also fill `proposed_canonical` and `domain` columns
4. Re-run this cell to compute Cohen's κ

In [ ]:
from sklearn.metrics import cohen_kappa_score

appendix_d_path = APPENDIX_DIR / 'appendix_d_novel_candidates.csv'
if appendix_d_path.exists():
    df_d = pd.read_csv(appendix_d_path)
    
    # Check if expert review has been filled in
    r1_filled = df_d['rater_1_assessment'].notna() & (df_d['rater_1_assessment'] != '')
    r2_filled = df_d['rater_2_assessment'].notna() & (df_d['rater_2_assessment'] != '')
    both_filled = r1_filled & r2_filled
    
    if both_filled.sum() >= 5:
        df_reviewed = df_d[both_filled]
        kappa = cohen_kappa_score(
            df_reviewed['rater_1_assessment'],
            df_reviewed['rater_2_assessment']
        )
        # Landis & Koch (1977) interpretation
        if kappa < 0.2:   interp = 'slight'
        elif kappa < 0.4: interp = 'fair'
        elif kappa < 0.6: interp = 'moderate'
        elif kappa < 0.8: interp = 'substantial'
        else:             interp = 'almost perfect'
        
        print(f"Cohen's κ = {kappa:.3f} ({interp} agreement, Landis & Koch 1977)")
        print(f'Based on {len(df_reviewed)} reviewed candidates')
        
        # Confirmed count
        confirmed = (df_reviewed['rater_1_assessment'] == 'confirmed').sum()
        fp        = (df_reviewed['rater_1_assessment'] == 'false_positive').sum()
        indet     = (df_reviewed['rater_1_assessment'] == 'indeterminate').sum()
        print(f'Rater 1 breakdown: {confirmed} confirmed, {fp} false positive, {indet} indeterminate')
        
        # Save updated file with kappa in metadata
        df_d.to_csv(appendix_d_path, index=False)
        print(f'\nAppendix D updated: {appendix_d_path}')
    else:
        print(f'Expert review not yet complete ({both_filled.sum()} rows filled).')
        print('Fill rater_1_assessment and rater_2_assessment columns, then re-run.')
else:
    print('Run cells 6 and 7 first to generate appendix_d_novel_candidates.csv')

## 9. Layer 3 detection examples (qualitative — Table 6)

In [ ]:
# Show concrete examples of what Layer 3 detects
# These become Table 6 in Section 6.2.3
print('=== Table 6 Preview: Layer 3 Detection Examples ===')
print('(Populate proposed_canonical and domain columns from your domain knowledge)')
print()

example_texts = {
    'demo_1': 'Mitochondrial membrane voltage collapse was observed under oxidative burden conditions.',
    'demo_2': 'Little RNA molecules demonstrated effective gene silencing in cell culture experiments.',
    'demo_3': 'Surface sound resonance was used to measure receptor binding kinetics.',
    'demo_4': 'The electron transport series showed dysfunction under hypoxic conditions.',
    'demo_5': 'Gradient descent achieved convergence after excessive repetitions of the training loop.',
}

table6_rows = []
for doc_id, text in example_texts.items():
    hits = detector.detect(text)
    for hit in hits[:1]:  # top hit per text
        table6_rows.append({
            'span':           hit['tortured'],
            'log_perplexity': f"{hit['log_perplexity']:.3f}",
            'confidence':     f"{hit['confidence']:.3f}",
            'context':        text[:80],
        })

if table6_rows:
    df_t6 = pd.DataFrame(table6_rows)
    print(df_t6.to_string(index=False))
else:
    print('No spans detected above threshold in demo texts.')
    print('This is expected if calibrated threshold is high — use real PubMed corpus.')

## 10. Summary: values to copy into Section 6

In [ ]:
print('===================================================================')
print('COPY THESE VALUES INTO SECTION 6.2 OF THE PAPER')
print('===================================================================')
print()
print(f'Layer 3 calibrated threshold τ         : {detector.perplexity_threshold:.4f}')
print(f'  (at {PERPLEXITY_THRESHOLD_PERCENTILE:.0f}th percentile of clean corpus spans)')
print()
print(f'Perplexity distribution:')
if 'clean_plls' in dir():
    print(f'  Clean spans — mean PLL                : {np.mean(clean_plls):.3f}')
    print(f'  Clean spans — std PLL                 : {np.std(clean_plls):.3f}')
if 'tortured_plls' in dir():
    print(f'  Tortured spans — mean PLL             : {np.mean(tortured_plls):.3f}')
    print(f'  Tortured spans — std PLL              : {np.std(tortured_plls):.3f}')
print()
print(f'Table 5 values:')
print(f'  Total papers screened                  : {novel_results["total_papers"]}')
print(f'  Papers with flagged spans              : {novel_results["papers_with_flags"]}')
print(f'  Flag rate                              : {novel_results["flag_rate"]:.1%}')
print(f'  Total candidate spans                  : {novel_results["total_candidates"]}')
print(f'  Unique candidate spans                 : {novel_results["unique_spans"]}')
print()
print('Remaining Table 5 values (expert review required):')
print('  Expert-confirmed tortured phrases       : [fill after reviewing appendix_d]')
print('  False positives                         : [fill after reviewing appendix_d]')
print('  Indeterminate                           : [fill after reviewing appendix_d]')
print("  Cohen's kappa                           : [run cell 8 after expert review]")